# 阶段一/二：特征工程

**实验目标：**
1. 完成数据清洗（缺失值处理、标识字段删除）
2. 完成基础特征提取（从文本字段中提取数值信息）
3. 构造高级衍生特征
4. 为建模准备数据

In [ ]:
# 环境初始化
import sys
sys.path.append('..')

from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_raw_data, save_processed_data, save_feature_data
from src.feature_engineering import full_feature_pipeline

# 设置目录
DATA_DIR = Path('..') / 'data'

print('✓ 环境初始化完成')

## 1. 加载原始数据

In [ ]:
# 加载原始数据
house = load_raw_data(DATA_DIR)
print(f'\n数据形状: {house.shape}')
display(house.head())

## 2. 执行特征工程流水线

In [ ]:
# 执行完整的特征工程流水线
house_clean = full_feature_pipeline(house)

## 3. 验证清洗结果

In [ ]:
# 验证清洗结果
print('=' * 50)
print('清洗结果验证')
print('=' * 50)
print(f'清洗后数据形状：{house_clean.shape}')
print(f'清洗后缺失值总数：{house_clean.isnull().sum().sum()}')
print()

# 新增特征统计
print('新增特征统计：')
new_features = ['室', '厅', '距离学校_米', '房龄', '楼层_数值',
                '单位面积房间数', '学校等级评分', '学校特色数量',
                '楼层比例', '每室平均面积', '房龄等级', '总价单价比']
available_new = [f for f in new_features if f in house_clean.columns]
display(house_clean[available_new].describe())

In [ ]:
# 二值特征分布
print('=' * 50)
print('二值特征分布')
print('=' * 50)
binary_cols = ['普通', '小班教学', '区重点', '体育类', '艺术类',
               '语文类', '双语', '科技类', '名校附属', '市重点', '外语类']
for col in binary_cols:
    if col in house_clean.columns:
        ones = house_clean[col].sum()
        total = len(house_clean)
        print(f'  {col:10s}：是={int(ones)} ({ones/total*100:.1f}%)，否={total-int(ones)} ({(total-ones)/total*100:.1f}%)')

## 4. 保存清洗后的数据

In [ ]:
# 保存清洗后的数据
save_processed_data(house_clean, DATA_DIR)

# 查看最终数据
print('\n最终数据列名：')
for i, col in enumerate(house_clean.columns, 1):
    print(f'  {i:2d}. {col} ({house_clean[col].dtype})')

## 阶段一/二特征工程总结

**完成的工作：**
1. 处理了所有缺失值（朝向用众数、建筑面积和建筑年代用中位数）
2. 删除了序号标识字段
3. 从户型中提取了室数和厅数
4. 从距离学校字段提取了数值距离
5. 构造了房龄特征（2026-建筑年代）
6. 对楼层进行了有序编码（低=1, 中=2, 高=3）
7. 对是/否二值特征进行了0/1编码

**新增的衍生特征：**
- 单位面积房间数：衡量空间利用率
- 学校等级评分：综合学校质量指标
- 学校特色数量：统计特色项目总数
- 楼层比例：所在楼层/总层数
- 每室平均面积：衡量房间宽敞程度
- 房龄等级：将连续房龄转为有序类别
- 总价单价比：反映装修/地段溢价程度